In [ ]:
%pip install lightautoml xgboost catboost

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 1.0/124.9 MB 6.3 MB/s eta 0:00:20
    --------------------------------------- 2.4/124.9 MB 6.1 MB/s eta 0:00:21
   - -------------------------------------- 3.7/124.9 MB 6.1 MB/s eta 0:00:21
   - -------------------------------------- 5.0/124.9 MB 6.2 MB/s eta 0:00:20
   - -------------------------------------- 6.0/124.9 MB 6.1 MB/s eta 0:00:20
   -- ------------------------------------- 7.3/124.9 MB 6.1 MB/s eta 0:00:20
   -- ------------------------------------- 8.7/124.9 MB 6.2 MB/s eta

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

TARGET_NAME = 'TARGET'

Train shape: (76020, 371)
Test shape: (75818, 370)


In [ ]:
X = df_train.drop(columns=[TARGET_NAME])
y = df_train[TARGET_NAME]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=RANDOM_STATE
)

print(f"Train size: {X_train.shape[0]}")
print(f"Val size: {X_val.shape[0]}")

Train size: 60816
Val size: 15204


In [ ]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

task = Task('binary')

roles = {
    'target': TARGET_NAME,
    'drop': ['SK_ID_CURR']
}

In [ ]:
automl_1 = TabularAutoML(
    task=task, 
    timeout=600,
    cpu_limit=4, 
    reader_params={'n_jobs': 4, 'cv': 5, 'random_state': RANDOM_STATE}
)

train_data = pd.concat([X_train, y_train], axis=1)

oof_pred_1 = automl_1.fit_predict(train_data, roles=roles, verbose=1)

val_pred_1 = automl_1.predict(X_val)

score_1 = roc_auc_score(y_val, val_pred_1.data[:, 0])
print(f"LAMA Config 1 ROC AUC: {score_1:.5f}")

[01:03:49] Stdout logging level is INFO.
[01:03:49] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[01:03:49] Task: binary

[01:03:49] Start automl preset with listed constraints:
[01:03:49] - time: 600.00 seconds
[01:03:49] - CPU: 4 cores
[01:03:49] - memory: 16 GB

[01:03:49] Train data shape: (60816, 371)

[01:04:00] Layer 1 train process start. Time left 588.93 secs
[01:04:03] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[01:04:15] Fitting Lvl_0_Pipe_0_Mod_0_LinearL2 finished. score = 0.7922651106354982
[01:04:15] Lvl_0_Pipe_0_Mod_0_LinearL2 fitting and predicting completed
[01:04:15] Time left 573.60 secs

[01:04:17] Selector_LightGBM fitting and predicting completed
[01:04:19] Start fitting Lvl_0_Pipe_1_Mod_0_LightGBM ...
[01:04:30] Fitting Lvl_0_Pipe_1_Mod_0_LightGBM finished. score = 0.8322119073144053
[01:04:30] Lvl_0_Pipe_1_Mod_0_LightGBM fitting and predicting completed
[01:04:30] Start hyperparameters optimization for Lvl

Optimization Progress:  34%|███▎      | 34/101 [01:27<02:53,  2.58s/it, best_trial=15, best_value=0.845]

[01:05:58] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM completed
[01:05:58] Start fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ...


[01:06:03] Fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM finished. score = 0.8361624792951139
[01:06:03] Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM fitting and predicting completed
[01:06:03] Start fitting Lvl_0_Pipe_1_Mod_2_CatBoost ...
[01:06:20] Fitting Lvl_0_Pipe_1_Mod_2_CatBoost finished. score = 0.8322708608265901
[01:06:20] Lvl_0_Pipe_1_Mod_2_CatBoost fitting and predicting completed
[01:06:20] Start hyperparameters optimization for Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost ... Time budget is 266.21 secs


Optimization Progress:  48%|████▊     | 48/101 [04:31<04:59,  5.66s/it, best_trial=45, best_value=0.847]

[01:10:52] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost completed
[01:10:52] Start fitting Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost ...


[01:11:39] Fitting Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost finished. score = 0.8375561339190402
[01:11:39] Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost fitting and predicting completed
[01:11:39] Time left 129.69 secs

[01:11:39] Layer 1 training completed.

[01:11:39] Blending: optimization starts with equal weights. Score = 0.8361265
[01:11:39] Blending: iteration 0: score = 0.8384287, weights = [0.         0.13328747 0.26099306 0.         0.60571945]
[01:11:40] Blending: iteration 1: score = 0.8385139, weights = [0.05308272 0.12011761 0.26143718 0.         0.56536245]
[01:11:40] Blending: iteration 2: score = 0.8385156, weights = [0.0560158  0.11775139 0.2925278  0.         0.533705  ]
[01:11:41] Blending: iteration 3: score = 0.8385187, weights = [0.05162313 0.1203589  0.29295573 0.         0.53506225]
[01:11:41] Blending: iteration 4: score = 0.8385190, weights = [0.05169772 0.12090626 0.29208145 0.         0.53531456]
[01:11:41] Blending: best score = 0.8385190, best weights = [0.05169772 0.12090

In [ ]:
automl_2 = TabularAutoML(
    task=task,
    timeout=600,
    cpu_limit=4,
    general_params={'use_algos': [['lgb', 'lgb_tuned', 'cb']]},
    reader_params={'n_jobs': 4, 'cv': 5, 'random_state': RANDOM_STATE}
)

oof_pred_2 = automl_2.fit_predict(train_data, roles=roles, verbose=1)
val_pred_2 = automl_2.predict(X_val)

score_2 = roc_auc_score(y_val, val_pred_2.data[:, 0])
print(f"LAMA Config 2 ROC AUC: {score_2:.5f}")

[01:11:53] Stdout logging level is INFO.
[01:11:53] Task: binary

[01:11:53] Start automl preset with listed constraints:
[01:11:53] - time: 600.00 seconds
[01:11:53] - CPU: 4 cores
[01:11:53] - memory: 16 GB

[01:11:53] Train data shape: (60816, 371)

[01:12:05] Layer 1 train process start. Time left 588.25 secs
[01:12:07] Selector_LightGBM fitting and predicting completed
[01:12:09] Start fitting Lvl_0_Pipe_0_Mod_0_LightGBM ...
[01:12:21] Fitting Lvl_0_Pipe_0_Mod_0_LightGBM finished. score = 0.8322119073144053
[01:12:21] Lvl_0_Pipe_0_Mod_0_LightGBM fitting and predicting completed
[01:12:21] Start hyperparameters optimization for Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM ... Time budget is 216.98 secs


Optimization Progress:  86%|████████▌ | 87/101 [03:37<00:34,  2.50s/it, best_trial=82, best_value=0.847]

[01:15:58] Hyperparameters optimization for Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM completed
[01:15:58] Start fitting Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM ...


[01:16:05] Fitting Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM finished. score = 0.8353441497551563
[01:16:05] Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM fitting and predicting completed
[01:16:05] Start fitting Lvl_0_Pipe_0_Mod_2_CatBoost ...
[01:16:23] Fitting Lvl_0_Pipe_0_Mod_2_CatBoost finished. score = 0.8322708608265901
[01:16:23] Lvl_0_Pipe_0_Mod_2_CatBoost fitting and predicting completed
[01:16:23] Time left 330.09 secs

[01:16:23] Layer 1 training completed.

[01:16:23] Blending: optimization starts with equal weights. Score = 0.8372216
[01:16:23] Blending: iteration 0: score = 0.8373714, weights = [0.18472187 0.48581994 0.32945818]
[01:16:23] Blending: no improvements for score. Terminated.

[01:16:23] Blending: best score = 0.8373714, best weights = [0.18472187 0.48581994 0.32945818]
[01:16:23] Automl preset training completed in 270.43 seconds

[01:16:23] Model description:
Final prediction for new objects (level 0) = 
	 0.18472 * (5 averaged models Lvl_0_Pipe_0_Mod_0_LightGBM) +
	 0.48582 *

In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'SK_ID_CURR' in numeric_features:
    numeric_features.remove('SK_ID_CURR')

print(f"Numeric features: {len(numeric_features)}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features)
    ])

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=4
)

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', xgb_model)])

Numeric features: 370
Categorical features: 0


In [ ]:
param_dist = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 5, 7],
    'classifier__subsample': [0.7, 0.8, 0.9],
    'classifier__colsample_bytree': [0.7, 0.8, 0.9]
}

search = RandomizedSearchCV(
    pipeline, 
    param_distributions=param_dist, 
    n_iter=10,
    scoring='roc_auc', 
    cv=3, 
    verbose=1, 
    random_state=RANDOM_STATE,
    n_jobs=4
)

print("Starting Hyperparameter Tuning...")
search.fit(X_train, y_train)

print(f"Best params: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.5f}")

best_model = search.best_estimator_

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


C:\Users\vanos\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xgboost\core.py:158: UserWarning: [00:48:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best params: {'classifier__subsample': 0.9, 'classifier__n_estimators': 200, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 0.8}
Best CV score: 0.83517


In [ ]:
y_pred_val = best_model.predict_proba(X_val)[:, 1]
custom_score = roc_auc_score(y_val, y_pred_val)

print(f"Custom Solution ROC AUC: {custom_score:.5f}")

print("-" * 30)
print("Final Results:")
print(f"LAMA Config 1: {score_1:.5f}")
print(f"LAMA Config 2: {score_2:.5f}")
print(f"Custom Solution: {custom_score:.5f}")

Custom Solution ROC AUC: 0.84742
------------------------------
Final Results:
LAMA Baseline: Skipped (Import Error)
Custom Solution: 0.84742


In [ ]:
if score_1 > score_2:
    best_lama = automl_1
    print("Best LAMA model: Config 1")
else:
    best_lama = automl_2
    print("Best LAMA model: Config 2")

test_pred_lama = best_lama.predict(df_test).data[:, 0]

submission_lama = pd.DataFrame({
    'SK_ID_CURR': df_test['SK_ID_CURR'] if 'SK_ID_CURR' in df_test.columns else df_test.index,
    'TARGET': test_pred_lama
})

submission_lama.to_csv('submission_lama.csv', index=False)
print("LAMA Submission saved to submission_lama.csv")

X_test = df_test.copy()
if 'TARGET' in X_test.columns:
    X_test = X_test.drop(columns=['TARGET'])

test_pred_custom = best_model.predict_proba(X_test)[:, 1]

submission_custom = pd.DataFrame({
    'SK_ID_CURR': df_test['SK_ID_CURR'] if 'SK_ID_CURR' in df_test.columns else df_test.index,
    'TARGET': test_pred_custom
})

submission_custom.to_csv('submission_custom.csv', index=False)
print("Custom Submission saved to submission_custom.csv")

Submission saved to submission_custom.csv
